# Step 1 基準線の測定（docs/v2-light-model-plan.md §4・Colab T4 用）

基準線: **sarashina2.2-3b・e5-large・リランク off** で G0 / Q / N / P / V を測り、基準線として記録する（合否も付ける）。データは架空の社内規程（evaluation/corpus）だけ。トークン不要（使うモデルは全て公開）。ランタイムは **T4 GPU**。

Step 0.5 の試運転（step05_trial.ipynb）との違い:

- 問は全部（50 問 + N 12 問 = 62 問）。8 問の抜き出しはしない
- GPU_UTIL は予算の式の値に固定する（8GB 0.50・12GB 0.76）。予算を超えても **下げて起動し直さない**（V は救わない。超過・起動しないはそのまま記録）
- 採点役は本物（`stelterlab/phi-4-AWQ`・14B の 4bit・候補と別の系統）。採点役の確認（正解の分かっている 5 例）に **通ったときだけ** 採点する。通らなければ採点せず理由を記録する（回答は残るので、採点役を替えて採点のセルだけ流し直せる）
- 採点した回で採点役の呼び出しが落ちた問（長すぎる要求を含む）が 1 問でもあれば、その回の Q は読まない（`judge_failures` に出る）
- 最後に G0（RAG 接続の確認・リランク off）を 12GB の設定で流す。止める試験で両サーバーを落とすので、採点の後に流し、P・V の区間には入れない

流れ（「Run all」で全部流れる。止まるのは準備の失敗だけ）:

1. 準備（仮想環境・Qdrant・GPU の記録）→ 索引
2. 測定: vLLM を 8192 / 0.85 で起動し、62 問を `LLM_MAX_TOKENS=1` で流して入力トークン数を取る → `max-model-len = ceil((最大 + 1024) / 512) × 512`
3. 予算ごと（8GB → 12GB）: GPU が空くのを待つ → GPU_UTIL = (予算 − 常駐) ÷ カード全体 → 起動 → 62 問を `--no-judge` で回答
4. 採点: 採点役を 8192 / 0.85 で別に起動 → 採点役の確認 → 通れば各予算の回答を `--judge-from`
5. G0 → まとめ（summary.json）→ zip

読み方:

- 予算ごとの `ok` は全問回答・最長の問・OOM なし・KV の追い出しなし・予算内。`None` は「取れなかった＝未確認」で、通ったことにはしない
- 10/22 の試運転では sarashina2.2-3b は 8GB で起動しなかった（最大 7,785 MiB）。8GB の V は不合格の見込みで、そのまま記録する
- 結果は `/content/step1/` に残る。最後のセルで zip にする（本文を含むので、リポジトリに入れるのは run_meta.json とまとめだけにする）


In [ ]:
# 依存の衝突を避けるため、vLLM と RAG 側を別の仮想環境に入れる（rag_connection_check.ipynb と同じ）
!pip -q install uv
!git clone -q --depth 1 -b redesign/v2 https://github.com/Maaa2005/local-rag-agent.git /content/repo
!uv venv -q /content/rag-env
# torch は CPU 版を先に入れる（同じ行で CPU の index を足すと uv が他の依存を解けない）
!uv pip install -q --python /content/rag-env/bin/python torch --index-url https://download.pytorch.org/whl/cpu
!uv pip install -q --python /content/rag-env/bin/python "transformers<5" sentence-transformers==3.3.1 \
    qdrant-client==1.12.1 aiosqlite==0.20.0 pydantic-settings==2.6.1 openai==1.57.0 httpx==0.28.1 pyyaml
!uv venv -q /content/vllm-env && uv pip install -q --python /content/vllm-env/bin/python vllm

In [ ]:
# 設定
# 基準線（§4 Step 1）: sarashina2.2-3b・e5-large・リランク off。設定は bfloat16 で、T4 は bf16 が使えないので float16 で動く
MODEL = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
EXTRA_ARGS = []
# 採点役: 14B の 4bit（AWQ）。候補（SB Intuitions・Liquid・Qwen・Gemma・llm-jp・NVIDIA）と別の系統。MIT
JUDGE_MODEL = "stelterlab/phi-4-AWQ"
JUDGE_EXTRA_ARGS = []
JUDGE_LEN, JUDGE_UTIL = 8192, 0.85       # 採点役は単独で起動する
MEASURE_LEN, MEASURE_UTIL = 8192, 0.85   # 入力トークン数を測る起動（測る前なので長めに取る）
RESIDENT_MIB = 0                          # GPU に載せた埋め込み・リランカーの MiB（CPU なので 0）
FAULT_TIMEOUT = 15                        # G0 の止める試験で待つ秒数
QDRANT_VER = "v1.12.6"                    # qdrant-client 1.12.1 に合わせる
REPO = "/content/repo"
RAG_PY = "/content/rag-env/bin/python"
VLLM_BIN = "/content/vllm-env/bin/vllm"
OUT = "/content/step1"                    # 結果の置き場所（やり直すときはセルを流す前に消すか名前を替える）
PORT = 8001
STEP1_NOTE = "Step 1 基準線（GPU_UTIL は式の値に固定・超えても下げない）"


In [ ]:
import json, os, pathlib, re, signal, subprocess, sys, threading, time, urllib.request

sys.path.insert(0, REPO)   # evaluation.colab.step05 を使う（標準ライブラリだけなので kernel から import できる）
from evaluation.colab import step05 as s5

OUTP = pathlib.Path(OUT)
if OUTP.exists() and any(OUTP.iterdir()):
    raise RuntimeError(f"{OUT} が空でない（前の結果と混ざるので止める）。消すか設定セルの OUT を替える")
OUTP.mkdir(parents=True, exist_ok=True)

def launch(argv, log, env=None):
    # 新しいプロセスグループで起動する（落とすときに killpg を使うため。pid = pgid）
    f = open(log, "w")
    return subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=env)

def wait(url, proc, log, limit):
    # 起動したら True。プロセスが落ちた・時間切れなら、ログの抜き出しを出して False（止めずに記録へ回す）
    t = time.time()
    while time.time() - t < limit:
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen(url, timeout=3)
            print("ready:", url, f"{time.time() - t:.0f} 秒")
            return True
        except Exception:
            time.sleep(3)
    lines = [l[:400] for l in pathlib.Path(log).read_text(errors="replace").splitlines()]
    hits = [i for i, l in enumerate(lines) if re.search(r"Traceback|ERROR|Error|Exception|Killed", l)]
    print("--- 起動しない:", url, "（全文は", log, "）---")
    print("\n".join(f"{i + 1}: {lines[i][:300]}" for i in hits[:40]) or "エラーらしい行なし")
    print("--- ログの末尾 30 行 ---")
    print("\n".join(lines[-30:]))
    return False

def kill(proc):
    if proc is None:
        return
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    proc.wait()

def read_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return s5.parse_smi_mib(out)

def fetch(url, timeout=5):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            return r.read().decode("utf-8", errors="replace")
    except Exception:
        return None

# Qdrant
qd = pathlib.Path("/content/qdrant"); (qd / "bin").mkdir(parents=True, exist_ok=True)
!curl -sSL https://github.com/qdrant/qdrant/releases/download/{QDRANT_VER}/qdrant-x86_64-unknown-linux-gnu.tar.gz | tar xz -C /content/qdrant/bin
q_env = dict(os.environ, QDRANT__STORAGE__STORAGE_PATH=str(qd / "storage"), QDRANT__SERVICE__HTTP_PORT="6333")
qdrant = launch([str(qd / "bin" / "qdrant")], "/content/qdrant.log", q_env)
if not wait("http://127.0.0.1:6333/readyz", qdrant, "/content/qdrant.log", 60):
    raise RuntimeError("Qdrant が起動しない（準備の失敗なのでここで止める）")

# V（GPU メモリ）: カード全体の使用量を 1 秒ごとに記録する（vLLM の確保分・GPU に載せた物を全部含む）
GPU_MEM = []   # (time.time(), MiB)
def _log_gpu_mem():
    while True:
        try:
            GPU_MEM.append((time.time(), read_mib()))
        except Exception:
            pass
        time.sleep(1)
if "GPU_MEM_THREAD" not in globals():
    GPU_MEM_THREAD = threading.Thread(target=_log_gpu_mem, daemon=True)
    GPU_MEM_THREAD.start()

GPU_NAME, CAP, TOTAL_MIB = [x.strip() for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.splitlines()[0].split(",")]
TOTAL_MIB = int(TOTAL_MIB)
VLLM_VER = subprocess.run(["/content/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                          capture_output=True, text=True).stdout.strip() or "不明"
BASELINE_MIB = read_mib()   # 何も載せていない時の使用量（GPU が空いたかの基準）
REV = s5.hf_revision(MODEL)
JUDGE_REV = s5.hf_revision(JUDGE_MODEL)
print(GPU_NAME, "compute", CAP, "全体", TOTAL_MIB, "MiB / 起動前", BASELINE_MIB, "MiB / vllm", VLLM_VER)
print("revision:", REV, "/ 採点役:", JUDGE_REV)

In [ ]:
# 1. 索引（別プロセス・vLLM の前）。DATA_DIR・QDRANT_URL・QDRANT_COLLECTION は index.json に残り、評価側はそこから作る
WORK = OUTP / "index_work"
COLLECTION = f"step1_{int(time.time())}"
p = subprocess.run([RAG_PY, f"{REPO}/evaluation/colab/corpus_index.py", "--repo", REPO, "--work", str(WORK),
                    "--qdrant-url", "http://127.0.0.1:6333", "--collection", COLLECTION],
                   cwd=REPO, capture_output=True, text=True)
print(p.stdout[-2000:], p.stderr[-3000:])
if p.returncode != 0:
    raise RuntimeError("索引に失敗（準備の失敗なのでここで止める）")
INDEX = json.loads((WORK / "index.json").read_text())
print(INDEX)

In [ ]:
# 2. 測定: 入力トークン数だけ取る（LLM_MAX_TOKENS=1 はこのプロセスだけ）
BASE_URL = f"http://127.0.0.1:{PORT}/v1"
WINDOWS = {}    # 区間名 -> (開始, 終了)。GPU の記録の切り出しに使う

def run_eval(argv, env, name, log):
    t0 = time.time()
    with open(log, "w") as f:
        rc = subprocess.run(argv, cwd=REPO, env=dict(os.environ, **env), stdout=f, stderr=subprocess.STDOUT).returncode
    WINDOWS[name] = (t0, time.time())
    tail = pathlib.Path(log).read_text(errors="replace").splitlines()[-8:]
    print(f"[{name}] rc={rc} {WINDOWS[name][1] - t0:.0f} 秒\n  " + "\n  ".join(tail))
    return rc

MEASURE = {"max_model_len": MEASURE_LEN, "gpu_util": MEASURE_UTIL}
s5.wait_gpu_released(read_mib, BASELINE_MIB)
vllm = launch(s5.vllm_argv(VLLM_BIN, MODEL, PORT, MEASURE_LEN, MEASURE_UTIL, REV.get("sha"), extra=EXTRA_ARGS),
              str(OUTP / "vllm_measure.log"))
if not wait(f"{BASE_URL}/models", vllm, str(OUTP / "vllm_measure.log"), 1200):
    raise RuntimeError("測定用の vLLM が起動しない（max-model-len が決まらないので止める）")
m = re.search(r"dtype=torch\.(\w+)", (OUTP / "vllm_measure.log").read_text(errors="replace"))
DTYPE = m.group(1) if m else "不明"
META_COMMON = {"step": "1", "gpu": GPU_NAME, "compute_cap": CAP, "model": MODEL, "model_revision": REV.get("sha"),
               "vllm": VLLM_VER, "dtype": DTYPE, "quant": "なし", "rerank": "off", "note": STEP1_NOTE}
rc = run_eval(s5.eval_cmd(RAG_PY, OUTP / "measure", meta={**META_COMMON, "purpose": "prompt_tokens",
                                                          "max_model_len": MEASURE_LEN, "gpu_util": MEASURE_UTIL,
                                                          "llm_max_tokens": 1}),
              s5.eval_env(INDEX["env"], BASE_URL, LLM_MAX_TOKENS="1"), "measure", str(OUTP / "measure.log"))
kill(vllm)
STATS = s5.prompt_token_stats(OUTP / "measure" / "raw.jsonl")   # 1 問でも取れなければ止まる
MAX_LEN = s5.needed_max_model_len(STATS["max_prompt_tokens"])
MEASURE.update(STATS, rc=rc, needed_max_model_len=MAX_LEN, dtype=DTYPE)
print("入力の最大", STATS["max_prompt_tokens"], "トークン（", STATS["max_id"], "・", STATS["n"], "問）→ max-model-len", MAX_LEN)
if MAX_LEN > MEASURE_LEN:
    print("注意: 必要な長さが測定の起動（", MEASURE_LEN, "）より長い。測定は通ったので値は使えるが、採点の起動も足りない可能性がある")

In [ ]:
# 3. 予算ごと（8GB → 12GB）に式の GPU_UTIL で起動して 62 問を全部。超えても下げない（V は救わない）
ALL_IDS = s5.dataset_ids(REPO)   # 重複があれば止まる
RUNS = {}   # "8GB" / "12GB" -> 結果

def run_budget(budget):
    budget_mib = s5.BUDGETS_MIB[budget]
    util = s5.gpu_util_for_budget(budget_mib, TOTAL_MIB, RESIDENT_MIB)
    rec = {"budget": budget, "budget_mib": budget_mib, "gpu_util": util, "max_model_len": MAX_LEN}
    s5.wait_gpu_released(read_mib, BASELINE_MIB)
    log = OUTP / f"vllm_{budget}.log"
    t_launch = time.time()
    proc = launch(s5.vllm_argv(VLLM_BIN, MODEL, PORT, MAX_LEN, util, REV.get("sha"), extra=EXTRA_ARGS), str(log))
    up = wait(f"{BASE_URL}/models", proc, str(log), 1200)
    WINDOWS[f"{budget}:launch"] = (t_launch, time.time())
    if not up:
        kill(proc)
        rec.update(started=False, peak_mib=s5.peak_in_window(GPU_MEM, t_launch, time.time())["peak_mib"])
        RUNS[budget] = rec
        print(f"[{budget}] 起動しない（GPU_UTIL {util}）。V は不合格として記録して先へ進む")
        return rec
    log_start = len(log.read_text(errors="replace"))
    before = s5.parse_prometheus_counter(fetch(f"http://127.0.0.1:{PORT}/metrics") or "")
    out_dir = OUTP / f"run_{budget}"
    rc = run_eval(s5.eval_cmd(RAG_PY, out_dir, meta={**META_COMMON, "budget": budget, "budget_mib": budget_mib,
                                                     "gpu_util": util, "max_model_len": MAX_LEN}),
                  s5.eval_env(INDEX["env"], BASE_URL), f"{budget}:run", str(OUTP / f"run_{budget}.log"))
    metrics_text = fetch(f"http://127.0.0.1:{PORT}/metrics") or ""
    after = s5.parse_prometheus_counter(metrics_text)
    log_scan = s5.scan_vllm_log(log.read_text(errors="replace")[log_start:])
    kill(proc)
    mem = s5.gpu_mem_record(GPU_MEM, {"launch": WINDOWS[f"{budget}:launch"], "run": WINDOWS[f"{budget}:run"]},
                            budget_mib, TOTAL_MIB, BASELINE_MIB)
    checks = s5.trial_checks(out_dir / "raw.jsonl", ALL_IDS, STATS["max_id"], mem,
                             s5.counter_delta(before, after), log_scan, note=STEP1_NOTE)
    if (out_dir / "run_meta.json").exists():
        # --judge-from は元の run_meta.json を採点の時点で写すので、採点の前に足す
        s5.attach_to_run_meta(out_dir, "gpu_mem", mem)
        s5.attach_to_run_meta(out_dir, "run_checks", checks)
        s5.attach_to_run_meta(out_dir, "vllm_log_scan", log_scan)
    rec.update(started=True, rc=rc, out_dir=str(out_dir), peak_mib=mem["peak_mib"], checks=checks,
               preempt_metric_lines=s5.metric_lines(metrics_text)[:10])
    RUNS[budget] = rec
    print(f"[{budget}] 最大 {mem['peak_mib']} MiB / 予算 {budget_mib} MiB → ok={checks['ok']}")
    # P（時間）を問ごとにセルの出力へ全部出す（本文は出さない。ランタイムが切れても数値が残る）
    if (out_dir / "raw.jsonl").exists():
        nums = s5.numbers_only(out_dir / "raw.jsonl")
        rec["times"] = nums["times"]
        print(f"P_{budget}_JSON " + json.dumps({"budget": budget, "checks": checks, "peak_mib": mem["peak_mib"], **nums},
                                                ensure_ascii=False))
    return rec

for budget in s5.BUDGETS_MIB:
    run_budget(budget)


In [ ]:
# 4. 採点: 採点役を単独で起動 → 採点役の確認 → 通ったときだけ各予算の回答を --judge-from
JUDGE = {"model": JUDGE_MODEL, "revision": JUDGE_REV.get("sha"), "role": "real-judge", "quant": "AWQ 4bit",
         "max_model_len": JUDGE_LEN, "gpu_util": JUDGE_UTIL}
s5.wait_gpu_released(read_mib, BASELINE_MIB)
judge_log = OUTP / "vllm_judge.log"
judge = launch(s5.vllm_argv(VLLM_BIN, JUDGE_MODEL, PORT, JUDGE_LEN, JUDGE_UTIL, JUDGE_REV.get("sha"),
                            extra=JUDGE_EXTRA_ARGS), str(judge_log))
JUDGE["started"] = wait(f"{BASE_URL}/models", judge, str(judge_log), 1800)
JUDGED = {}
if JUDGE["started"]:
    env = dict(os.environ, **s5.eval_env(INDEX["env"], BASE_URL))
    pre_out = OUTP / "judge_precheck.json"
    p = subprocess.run([RAG_PY, "-m", "evaluation.judge_precheck", "--out", str(pre_out)],
                       cwd=REPO, env=env, capture_output=True, text=True)
    print(p.stdout[-3000:], p.stderr[-3000:])
    # 落ちて JSON が無いときも未判定（judge_precheck の終了コード 0=pass / 1=fail / 3=未判定）
    JUDGE["precheck"] = json.loads(pre_out.read_text())["result"] if pre_out.exists() else "未判定"
    JUDGE["precheck_rc"] = p.returncode
    if pre_out.exists():
        # 採点役の確認の全部（5 例の点・根拠コメント）をセルの出力に出す。fixture は架空の規程なので出してよい
        print("PRECHECK_JSON " + json.dumps(json.loads(pre_out.read_text()), ensure_ascii=False))
    if JUDGE["precheck"] != "pass":
        JUDGE["skipped"] = f"採点役の確認が {JUDGE['precheck']} なので採点しない（回答は run_* に残る）"
        print(JUDGE["skipped"])
    else:
        for label, rec in RUNS.items():
            raw = pathlib.Path(rec.get("out_dir", "")) / "raw.jsonl"
            if not rec.get("started") or not raw.exists():
                continue
            out_dir = OUTP / f"judged_{label}"
            rc = run_eval(s5.judge_cmd(RAG_PY, raw, out_dir, meta={"step": "1", "judge_model": JUDGE_MODEL,
                                                                   "judge_revision": JUDGE["revision"],
                                                                   "judge_role": JUDGE["role"],
                                                                   "judge_quant": JUDGE["quant"],
                                                                   "judge_precheck": JUDGE["precheck"]}),
                          s5.eval_env(INDEX["env"], BASE_URL), f"judge:{label}", str(OUTP / f"judged_{label}.log"))
            # 採点役の呼び出しが落ちた問が 1 問でもあれば、その回の Q は読まない
            fails = s5.judge_failures(out_dir / "raw.jsonl")
            JUDGED[label] = {"rc": rc, "out_dir": str(out_dir), **fails, "q_readable": rc == 0 and not fails["judge_failed"]}
            print(f"[judge:{label}] rc={rc} 採点 {fails['n_rows']} 問 / 採点役の失敗 {len(fails['judge_failed'])} 問")
            if (out_dir / "raw.jsonl").exists():
                print(f"Q_{label}_JSON " + json.dumps(s5.numbers_only(out_dir / "raw.jsonl"), ensure_ascii=False))
else:
    JUDGE["precheck"] = "未判定"
    JUDGE["skipped"] = "採点役が起動しない（回答は run_* に残る。採点役を替えてこのセルだけ流し直せる）"
    print(JUDGE["skipped"])
kill(judge)
print(JUDGE)


In [ ]:
# 5. G0（RAG 接続の確認・リランク off）。止める試験で両サーバーを落とすので最後に流す（P・V の区間には入れない）
import shutil
G0 = {"rerank": "off", "fault_timeout": FAULT_TIMEOUT}
if RUNS.get("12GB", {}).get("started"):
    s5.wait_gpu_released(read_mib, BASELINE_MIB)
    G0.update(budget="12GB", gpu_util=RUNS["12GB"]["gpu_util"], max_model_len=MAX_LEN)
    g0_log = OUTP / "vllm_g0.log"
    vllm = launch(s5.vllm_argv(VLLM_BIN, MODEL, PORT, MAX_LEN, G0["gpu_util"], REV.get("sha"), extra=EXTRA_ARGS),
                  str(g0_log))
    if wait(f"{BASE_URL}/models", vllm, str(g0_log), 1200):
        WORK_G0 = OUTP / "g0_work"
        p = subprocess.run([RAG_PY, f"{REPO}/evaluation/colab/rag_connection_check.py", "--repo", REPO,
                            "--work", str(WORK_G0), "--qdrant-url", "http://127.0.0.1:6333", "--vllm-url", BASE_URL,
                            "--qdrant-pgid", str(qdrant.pid), "--vllm-pgid", str(vllm.pid),
                            "--fault-timeout", str(FAULT_TIMEOUT), "--rerank", "off",
                            "--label", f"colab-{GPU_NAME.split()[-1]}-step1-rerank-off"],
                           cwd=REPO, capture_output=True, text=True)
        (OUTP / "g0.log").write_text(p.stdout + "\n--- stderr ---\n" + p.stderr)
        print(p.stdout[-4000:], p.stderr[-3000:])
        res = WORK_G0 / "rag_check_result.json"
        G0["rc"] = p.returncode
        G0["ok"] = json.loads(res.read_text()).get("ok") if res.exists() else None   # 結果が無ければ未確認
        if res.exists():
            shutil.copy(res, OUTP / "g0_result.json")
    else:
        G0["skipped"] = "G0 用の vLLM が起動しない"
    kill(vllm)
else:
    G0["skipped"] = "12GB で起動しなかったので G0 は流さない"
print("G0:", G0)


In [ ]:
# 6. まとめ（Step 1 基準線）
rows = []
for label, rec in RUNS.items():
    c = rec.get("checks") or {}
    j = JUDGED.get(label, {})
    rows.append({"label": label, "gpu_util": rec["gpu_util"], "started": rec["started"], "peak_mib": rec.get("peak_mib"),
                 "budget_mib": rec["budget_mib"], **{k: c.get(k) for k in
                 ("all_answered", "longest_ok", "no_oom", "no_preemption", "within_budget", "ok", "missing")},
                 "times": rec.get("times"), "judged_rc": j.get("rc"), "judge_failed": j.get("judge_failed"), "q_readable": j.get("q_readable")})
keys = list(rows[0]) if rows else []
print(" | ".join(keys))
for r in rows:
    print(" | ".join(str(r[k]) for k in keys))
SUMMARY = {"step": "1", "note": STEP1_NOTE, "gpu": GPU_NAME, "compute_cap": CAP, "total_mib": TOTAL_MIB,
           "baseline_mib": BASELINE_MIB, "vllm": VLLM_VER, "model": MODEL, "model_revision": REV, "dtype": DTYPE,
           "rerank": "off", "measure": MEASURE, "n_questions": len(ALL_IDS), "runs": rows, "judge": JUDGE,
           "judged": JUDGED, "g0": G0}
(OUTP / "summary.json").write_text(json.dumps(SUMMARY, ensure_ascii=False, indent=2) + "\n")
print("SUMMARY_JSON " + json.dumps(SUMMARY, ensure_ascii=False))
print("→", OUTP / "summary.json")


In [ ]:
# サーバーを落とし、結果を zip にする（回答の本文を含む。リポジトリに入れるのは run_meta.json と summary.json だけ）
kill(qdrant)
!cd /content && zip -qr step1_results.zip step1 -x 'step1/index_work/*' 'step1/g0_work/*'
!ls -la /content/step1_results.zip
